## Legal-model extension runner

Runs **one arm** of the {Llama, Mistral, Saul} × {zero-shot, QLoRA} grid on Kaggle by calling
`scripts/legal_model_extension.py run`. All prompt, training, decoding and scoring logic lives in
that script (design: [docs/extension/DESIGN.md](docs/extension/DESIGN.md)); this notebook only
installs the pinned stack, resolves inputs and launches it.

**Per run, edit only the parameters cell**, save, then push from the repo root:

```powershell
.\kaggle\run.ps1 -KernelDir kaggle\extension -OutDir kaggle_output_extension -Wait
```

Kaggle inputs (private datasets, see `kaggle/push_extension_datasets.ps1`): `legal-extension-code`,
`legal-extension-data`, `llama-adapters`, `hf-token`. Outputs land in
`/kaggle/working/runs/<model>_t<task>_<mode>[_smoke]/`.

In [ ]:
# Pin to ONE T4 before torch is imported anywhere: with both GPUs visible, device_map spreads the
# model over cuda:0/cuda:1 and TRL's loss fails with a cross-device indexing error.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# --- Environment config: run unchanged locally OR on Kaggle ---
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    DATA_DIR = Path("/kaggle/input")          # all attached datasets, read-only
    WORK_DIR = Path("/kaggle/working")        # the only writable + persisted dir
else:
    DATA_DIR = Path(os.getenv("DATA_DIR", "."))   # repo root: scripts/, cuad/, ledgar/, kaggle_output_*/
    WORK_DIR = Path(".")


def find_input(filename):
    """Locate an attached input file by name. Kaggle mounts each dataset under its own folder
    (the exact layout has varied), so search instead of hard-coding the mount path."""
    if not ON_KAGGLE:
        local = {"legal_model_extension.py": "scripts", "task1_metrics.py": "scripts",
                 "requirements-kaggle.txt": ".", "labels.json": "data/LEDGAR"}
        if filename in local:
            return DATA_DIR / local[filename] / filename
    matches = sorted(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {DATA_DIR} — is its dataset attached?")
    return matches[0]


print(f"ON_KAGGLE={ON_KAGGLE} | DATA_DIR={DATA_DIR} | WORK_DIR={WORK_DIR}")

In [ ]:
# --- Install the pinned GPU stack (same pins as the Llama notebooks) and record the environment ---
import subprocess, sys

CODE_DIR = find_input("legal_model_extension.py").parent
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(find_input("requirements-kaggle.txt"))], check=True)

# Full freeze of the environment the run actually used (plan Phase 2: pin `datasets` from this).
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(WORK_DIR / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
print(f"Code from {CODE_DIR}; wrote {WORK_DIR / 'pip_freeze.txt'} ({len(freeze.splitlines())} packages)")

In [ ]:
# --- HF token: private `hf-token` dataset -> Kaggle Secrets -> local .env. Never printed. ---
def get_hf_token():
    if ON_KAGGLE:
        for p in Path("/kaggle/input").rglob("hf_token.txt"):
            tok = p.read_text(encoding="utf-8").strip()
            if tok:
                return tok
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            return None
    from dotenv import load_dotenv
    load_dotenv(".env")
    return os.getenv("HF_TOKEN")


hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (hf-token dataset, Kaggle Secret, or local .env)"
os.environ["HF_TOKEN"] = hf_token   # inherited by the harness subprocess
print("HF token loaded.")

In [ ]:
# ============================ PARAMETERS — the only values edited per run ============================
TASK = 1              # 1 = CUAD risk clauses (T2/T3 not yet registered in the harness)
MODEL = "llama"       # "llama" | "mistral" | "saul"
MODE = "adapter"      # "baseline" (zero-shot) | "finetune" (QLoRA, then eval) | "adapter" (eval existing)
ADAPTER = 1           # mode "adapter" only: task number of the Llama adapter in `llama-adapters`, e.g. 1
LIMIT = None          # smoke test: evaluate only the first N validation examples; None = all
TRAIN_LIMIT = None    # smoke test: fine-tune on only the first N train examples; None = all
MAX_SEQ_LEN = 1024
# =====================================================================================================

In [ ]:
# --- Map TASK -> input files; rebuild the adapter folder if needed ---
import shutil

TASK_FILES = {
    1: {"train": "cuad_train.jsonl", "val": "cuad_validation.jsonl"},
    2: {"train": "cuad_task2_train.jsonl", "val": "cuad_task2_validation.jsonl"},
    3: {"train": "ledgar_task3_train.jsonl", "val": "ledgar_task3_validation.jsonl", "labels": "labels.json"},
}
LOCAL_DATA = {  # repo copies (CRLF on Windows checkouts — the harness hashes them LF-normalised)
    "cuad_train.jsonl": "cuad/train", "cuad_validation.jsonl": "cuad/validation",
    "cuad_task2_train.jsonl": "cuad/train", "cuad_task2_validation.jsonl": "cuad/validation",
    "ledgar_task3_train.jsonl": "ledgar/train", "ledgar_task3_validation.jsonl": "ledgar/validation",
}


def data_file(name):
    return find_input(name) if ON_KAGGLE else DATA_DIR / LOCAL_DATA[name] / name


files = TASK_FILES[TASK]
train_file = data_file(files["train"]) if MODE == "finetune" else None
val_file = data_file(files["val"])

adapter_dir = None
if MODE == "adapter":
    assert ADAPTER is not None, "MODE='adapter' needs ADAPTER = <task number of the Llama adapter>"
    # The dataset is flat (taskN__adapter_*.json/.safetensors); PEFT wants a folder. Rebuild it in
    # /tmp, NOT under WORK_DIR, so 55 MB of input is not re-exported as output.
    adapter_dir = Path("/tmp" if ON_KAGGLE else WORK_DIR) / f"llama_adapter_task{ADAPTER}"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    for name in ("adapter_config.json", "adapter_model.safetensors"):
        shutil.copy(find_input(f"task{ADAPTER}__{name}"), adapter_dir / name)

smoke = "_smoke" if (LIMIT or TRAIN_LIMIT) else ""
RUN_NAME = f"{MODEL}_t{TASK}_{MODE}{f'_adapter{ADAPTER}' if MODE == 'adapter' else ''}{smoke}"
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_NAME={RUN_NAME}\n  train={train_file}\n  val={val_file}\n  adapter={adapter_dir}")

In [ ]:
# --- Launch the harness; stream its output here AND into RUN_DIR/run.log ---
cmd = [sys.executable, "-u", str(CODE_DIR / "legal_model_extension.py"), "run",
       "--task", str(TASK), "--model", MODEL, "--mode", MODE,
       "--val_file", str(val_file), "--out_dir", str(RUN_DIR), "--max_seq_len", str(MAX_SEQ_LEN)]
if train_file:
    cmd += ["--train_file", str(train_file)]
if adapter_dir:
    cmd += ["--adapter_dir", str(adapter_dir)]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if TRAIN_LIMIT:
    cmd += ["--train_limit", str(TRAIN_LIMIT)]
print(" ".join(cmd), flush=True)

with open(RUN_DIR / "run.log", "w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
    returncode = proc.wait()
print(f"\nHarness exit code: {returncode}")

In [ ]:
# --- Keep the run self-contained for retrieval, then fail the kernel if the harness failed ---
shutil.copy(WORK_DIR / "pip_freeze.txt", RUN_DIR / "pip_freeze.txt")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(WORK_DIR)}  ({p.stat().st_size:,} bytes)")
assert returncode == 0, f"Harness failed (exit {returncode}) — see {RUN_DIR / 'run.log'}"